In [ ]:
import deeptime as dpt
from tqdm.notebook import tqdm # for progress bar
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import BoundaryNorm
import pyemma as pem
import pickle
import os


# # ----------------------------
# Nature-like defaults (apply once)
# ----------------------------
BASE_FONTSIZE = 8
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": BASE_FONTSIZE,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.labelsize": BASE_FONTSIZE,
    "axes.titlesize": BASE_FONTSIZE,
    "axes.linewidth": 0.8,
    "axes.spines.right": False,
    "axes.spines.top": False,

    "xtick.labelsize": BASE_FONTSIZE,
    "ytick.labelsize": BASE_FONTSIZE,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    "figure.dpi": 300,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
})


# ----------------------------
# Figure: two panels + top colorbars (Nature style)
# ----------------------------
def plot_qg_msm_nature(
    raw_data_all,
    meta_dtrajs_flat,
    nbins=100,
):
    # Data
    x = np.asarray(raw_data_all[:, 0])
    y = np.asarray(raw_data_all[:, 1])

    # Double-column width figure (two panels side-by-side)
    fig = plt.figure(figsize=(7.0, 3.15))  # ~180 mm wide, compact height
    gs = fig.add_gridspec(
        nrows=2, ncols=2,
        height_ratios=[0.10, 1.0],   # slim colorbar row
        width_ratios=[1, 1],
        hspace=0.15,
        wspace=0.20
    )

    # Axes
    cax_fe = fig.add_subplot(gs[0, 0])
    cax_sm = fig.add_subplot(gs[0, 1])
    ax_fe  = fig.add_subplot(gs[1, 0])
    ax_sm  = fig.add_subplot(gs[1, 1])

    # ---- Panel A: Free energy surface ----
    _, _, misc_fe = pem.plots.plot_free_energy(
        x, y,
        ax=ax_fe,
        nbins=nbins,
        cax=cax_fe,
        cbar_orientation="horizontal",
        cmap="coolwarm",
        levels=100,
        legacy=False
    )

    cb1 = misc_fe["cbar"]
    cb1.ax.xaxis.set_ticks_position("top")
    cb1.ax.xaxis.set_label_position("top")
    cb1.set_label(r"$-\ln(P)\,/\,k_{\mathrm{B}}T$")
    cb1.set_ticks([0, 3, 6, 9])
    cb1.outline.set_linewidth(0.8)

    # ---- Panel B: State map ----
    _, _, misc_sm = pem.plots.plot_state_map(
        x, y, meta_dtrajs_flat,
        ax=ax_sm,
        nbins=nbins,
        cax=cax_sm,
        cbar_orientation="horizontal",
    )

    cb2 = misc_sm["cbar"]
    cb2.ax.xaxis.set_ticks_position("top")
    cb2.ax.xaxis.set_label_position("top")
    cb2.set_label("Macrostate")
    cb2.outline.set_linewidth(0.8)
    labels = sorted(np.unique(meta_dtrajs_flat)+1)
    cb2.set_ticklabels(labels)

    # Shared axis formatting
    # y ticks every 0.02
    y_min = 0.0
    y_max = np.max(y)

    yticks = np.linspace(y_min, y_max, 8)
    # yticks = np.arange(0.00, 0.10, 0.02)  # covers up to 0.08–0.09 nicely
    for ax in (ax_fe, ax_sm):
        ax.set_xlabel(r"$Q$")
        ax.set_ylabel(r"$G$")
        ax.set_ylim(np.min(y), np.max(y))

        ax.set_yticks(yticks)
        ax.set_yticklabels([f"{v:.2f}" for v in yticks])  # optional, nice formatting
        ax.tick_params(top=False, right=False)

    ax_sm.set_ylabel("")   # remove ylabel on statemap

    return fig


# ----------------------------
# Usage
# ----------------------------
# plot_qg_msm_nature(raw_data_all, meta_dtrajs_flat, nbins=100, outbase="QG_MSM")

def plot_qg_binned(
    raw_data_all,
    meta_dtrajs_flat,
    bins=(80, 40),                 # (Q bins, G bins); reduce G bins if very sparse
    q_range=(0.0, 1.0),
    g_range=None,
    min_count=1,                   # display every sampled bin by default
    max_free_energy=12.0,          # cap -ln(P/Pmax) for readable colors
    fe_cmap="turbo",
    state_cmap="turbo",
    figsize=(7.0, 3.15),
):
    """
    Plot Q-G free energy and metastable-state maps using explicit rectangular bins.

    Free energy:
        F_i = -ln(P_i / P_max)

    State assignment:
        Each occupied bin receives the most frequently observed macrostate.
        Empty bins are left white.
    """
    xy = np.asarray(raw_data_all, dtype=float)
    states = np.asarray(meta_dtrajs_flat).reshape(-1)

    if xy.ndim != 2 or xy.shape[1] < 2:
        raise ValueError("raw_data_all must have shape (n_frames, 2).")
    if len(xy) != len(states):
        raise ValueError(
            f"Length mismatch: {len(xy)} Q-G points but {len(states)} state labels."
        )

    # Remove invalid observations
    valid = np.isfinite(xy[:, 0]) & np.isfinite(xy[:, 1])
    if np.issubdtype(states.dtype, np.number):
        valid &= np.isfinite(states)

    q = xy[valid, 0]
    g = xy[valid, 1]
    states = states[valid]

    if g_range is None:
        g_pad = max(0.02 * np.ptp(g), 1e-6)
        g_range = (max(0.0, g.min() - g_pad), g.max() + g_pad)

    nq, ng = bins
    q_edges = np.linspace(q_range[0], q_range[1], nq + 1)
    g_edges = np.linspace(g_range[0], g_range[1], ng + 1)

    # ---------------------------------------------------------
    # Put every observation into an explicit Q-G rectangular bin
    # ---------------------------------------------------------
    q_bin = np.searchsorted(q_edges, q, side="right") - 1
    g_bin = np.searchsorted(g_edges, g, side="right") - 1

    # Include values exactly on the upper boundary
    q_bin[q == q_edges[-1]] = nq - 1
    g_bin[g == g_edges[-1]] = ng - 1

    inside = (
        (q_bin >= 0) & (q_bin < nq) &
        (g_bin >= 0) & (g_bin < ng)
    )
    q_bin = q_bin[inside]
    g_bin = g_bin[inside]
    states = states[inside]

    # Frame counts in each bin; array orientation is [G, Q]
    counts = np.zeros((ng, nq), dtype=int)
    np.add.at(counts, (g_bin, q_bin), 1)

    occupied = counts >= min_count
    if not np.any(occupied):
        raise ValueError("No bins satisfy min_count within the selected ranges.")

    # Free energy relative to the most populated bin
    free_energy = np.full((ng, nq), np.nan)
    free_energy[occupied] = -np.log(
        counts[occupied] / counts[occupied].max()
    )
    free_energy[occupied] = np.minimum(
        free_energy[occupied], max_free_energy
    )

    # ---------------------------------------------------------
    # Assign each bin to its most frequently sampled macrostate
    # ---------------------------------------------------------
    state_values = np.sort(np.unique(states))
    state_to_index = {state: i for i, state in enumerate(state_values)}

    state_counts = np.zeros((len(state_values), ng, nq), dtype=int)
    for state, state_index in state_to_index.items():
        selected = states == state
        np.add.at(
            state_counts[state_index],
            (g_bin[selected], q_bin[selected]),
            1
        )

    dominant_state = np.full((ng, nq), np.nan)
    dominant_state[occupied] = np.argmax(
        state_counts[:, occupied], axis=0
    )

    # Colormaps: empty bins are white
    fe_cmap_obj = mpl.colormaps[fe_cmap].copy()
    fe_cmap_obj.set_bad("white")

    nstates = len(state_values)
    state_colors = mpl.colormaps[state_cmap](
        np.linspace(0.02, 0.98, nstates)
    )
    state_cmap_obj = mpl.colors.ListedColormap(state_colors)
    state_cmap_obj.set_bad("white")
    state_norm = BoundaryNorm(
        np.arange(-0.5, nstates + 0.5, 1),
        nstates
    )

    # ---------------------------------------------------------
    # Figure layout
    # ---------------------------------------------------------
    fig = plt.figure(figsize=figsize)
    grid = fig.add_gridspec(
        2, 2,
        height_ratios=[0.10, 1.0],
        hspace=0.15,
        wspace=0.20
    )

    cax_fe = fig.add_subplot(grid[0, 0])
    cax_state = fig.add_subplot(grid[0, 1])
    ax_fe = fig.add_subplot(grid[1, 0])
    ax_state = fig.add_subplot(grid[1, 1])

    # pcolormesh displays each bin as an explicit rectangle
    fe_plot = ax_fe.pcolormesh(
        q_edges,
        g_edges,
        np.ma.masked_invalid(free_energy),
        cmap=fe_cmap_obj,
        vmin=0,
        vmax=max_free_energy,
        shading="flat",
        rasterized=True
    )

    state_plot = ax_state.pcolormesh(
        q_edges,
        g_edges,
        np.ma.masked_invalid(dominant_state),
        cmap=state_cmap_obj,
        norm=state_norm,
        shading="flat",
        rasterized=True
    )

    # Top horizontal colorbars
    fe_cb = fig.colorbar(
        fe_plot, cax=cax_fe, orientation="horizontal"
    )
    fe_cb.ax.xaxis.set_ticks_position("top")
    fe_cb.ax.xaxis.set_label_position("top")
    fe_cb.set_label(r"$-\ln(P)/\,k_{\mathrm{B}}T$")
    fe_cb.outline.set_linewidth(0.8)

    state_cb = fig.colorbar(
        state_plot,
        cax=cax_state,
        orientation="horizontal",
        ticks=np.arange(nstates)
    )
    state_cb.ax.xaxis.set_ticks_position("top")
    state_cb.ax.xaxis.set_label_position("top")
    state_cb.ax.minorticks_off()
    state_cb.ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    state_cb.set_label("Macrostate")

    # Add 1 only when the trajectory labels are conventional zero-based integers
    zero_based = (
        np.issubdtype(state_values.dtype, np.integer) and
        np.array_equal(state_values, np.arange(nstates))
    )
    displayed_labels = state_values + 1 if zero_based else state_values
    state_cb.set_ticklabels([f"{x}" for x in displayed_labels])
    state_cb.outline.set_linewidth(0.8)

    for ax in (ax_fe, ax_state):
        ax.set_xlim(q_range)
        ax.set_ylim(g_range)
        ax.set_xlabel(r"$Q$")
        ax.set_ylabel(r"$G$")
        ax.tick_params(top=False, right=False)

    ax_state.set_ylabel("")
    # ax_fe.set_title("Free energy", pad=5)
    # ax_state.set_title("Metastable states", pad=5)

    return fig




# Step 1: Load your Q-G data
data = np.load("QG.npy")


# Test how many percent of variation that Q and G account for
scaler = StandardScaler()
raw_data_all = np.concatenate(data, axis=0)
scaler.fit(raw_data_all)
processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
conc_processed_data = np.concatenate(processed_data, axis=0)

tica = dpt.decomposition.TICA(lagtime=1, dim=None)
tica_model = tica.fit(processed_data)
projected = tica_model.transform(processed_data)

print(f"cummulative kinetic variance: {tica_model.model.cumulative_kinetic_variance}")

In [ ]:

"""
In general, I want to do MSM on QG space. However, some proteins do not sample the entangled state.
Hence, in that case we need to do MSM on Q space alone.
By defaults, n_dims is 2- mean perform on QG space. If not working, then set to n_dims=1 to use Q only
"""
# Set the number of dimensions to use for the MSM
n_dims = 2

if n_dims == 2: 
    # Step 2: Standardize the data
    scaler = StandardScaler()
    raw_data_all = np.concatenate(data, axis=0)
    scaler.fit(raw_data_all)
    processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
    conc_processed_data = np.concatenate(processed_data, axis=0)

elif n_dims == 1:
    # Incase MSM is not working on GQ space since the G is not significant, did MSM on Q space alone
    # Step 2: Load your Q-G data
    data = data[:, :, 0][:, :, np.newaxis]  # shape becomes (50, 13333, 1)
    # Step 2: Standardize the data
    scaler = StandardScaler()
    raw_data_all = np.concatenate(data, axis=0)
    scaler.fit(raw_data_all)
    processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
    conc_processed_data = np.concatenate(processed_data, axis=0)

In [ ]:
# Use KMeans from deeptime 
n_clusters = 100
kmeans_model = dpt.clustering.KMeans(n_clusters=n_clusters,
                                    max_iter=500,
                                    fixed_seed=True,
                                    progress=tqdm
                                    ).fit_fetch(processed_data)

# Transform each trajectory to discrete state trajectory
discrete_trajectories = [kmeans_model.transform(traj) for traj in processed_data]

# Get cluster centers (in scaled space)
cluster_centers_scaled = kmeans_model.cluster_centers

# Inverse transform to original Q-G scale
cluster_centers_original = scaler.inverse_transform(cluster_centers_scaled)

# Optional: Check shapes and values
print("Number of trajectories:", len(discrete_trajectories))
print("Shape of first discrete trajectory:", discrete_trajectories[0].shape)
# print("Unique states in first trajectory:", np.unique(discrete_trajectories[0]))

In [ ]:
# Choose a lag time (in frames) — determined from the implied timescales plot.
# currently, we use the lagtime = 1, which we do not care about the kinetics of the system.
lagtime = 1  # or 20, etc.

# Fit MSM
"""
The parameters allow_disconnected=False means that the MSM is not allowed to have disconnected states (default value in deeptime).
If set to True, the MSM will have disconnected states because the transition matrix may have disconnected and transient states,
and the estimated stationary distribution is only meaningful on the respective connected set.
"""
msm = dpt.markov.msm.MaximumLikelihoodMSM(lagtime=lagtime, allow_disconnected=False).fit(discrete_trajectories).fetch_model()

# Print some basic info
print("Number of microstates:", msm.transition_matrix.shape[0]) #equivalent to msm.n_states
print("Top 5 eigenvalues:", msm.eigenvalues()[:5])
print("Top 5 relaxation timescales:", msm.timescales()[:5])

# Print some basic info
print("Top 10 eigenvalues and timescales:")
print("Top 10 eigenvalues:", msm.eigenvalues()[:10])
print("Top 10 relaxation timescales:", msm.timescales()[:10])

# Plot the FE
nbins=30

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(3, 2))

# Plot the global landscape
pem.plots.plot_free_energy(raw_data_all[:, 0], raw_data_all[:, 1], nbins=nbins, ax=ax)
ax.set_ylim([0, np.max(raw_data_all[:, 1])])
ax.set_xlim([0, 1])
ax.set_title("Global")

plt.tight_layout()
plt.show()



In [ ]:
# Perform PCCA
n_macrostates = 4 # can be determined from the implied timescales plot.
pcca = msm.pcca(n_macrostates)

# Hard assignment: each cluster assigned to a macrostate
macrostate_assignment = pcca.assignments  # shape (n_microstates,)
print("Macrostate assignments (first 10):", macrostate_assignment[:10])

# Soft assignment: probability each microstate belongs to each macrostate
membership = pcca.memberships  # shape (n_microstates, n_macrostates)
print("Membership matrix shape:", membership.shape)

"""
Incase you want to see the assignment of the microstates to the macrostates
The assignment of the microstates to the macrostates
for each cluster from Kmeans clustering (order in the array, which macrostate it belongs to (value in the array))

The pcca.sets is the assignment of the macrostates to the macrostates
for each macrostate from PCCA (order in the array, which macrostate it belongs to (value in the array))
"""
print(f"The assignment of the microstates to the macrostates: {macrostate_assignment}")
print("The assignment of the macrostates to the macrostates:")
for idx, pcca_set in enumerate(pcca.sets):
    print(f"Macrostate {idx}: {pcca_set}")



In [ ]:

# Identify the core microstate of each macrostate:
# the microstate with the highest PCCA membership probability
most_prob_microstate = np.argmax(membership, axis=0)

# Obtain the Q and G coordinates of each macrostate's core microstate
core_coordinates = cluster_centers_original[most_prob_microstate]
core_q = core_coordinates[:, 0]
core_g = core_coordinates[:, 1]

print("Core microstate coordinates:")
for macro_idx, micro_idx in enumerate(most_prob_microstate):
    print(
        f"Macrostate {macro_idx}: "
        f"core microstate {micro_idx}, "
        f"Q={core_q[macro_idx]:.3f}, "
        f"G={core_g[macro_idx]:.4f}"
    )

# Native-state candidate selection
Q_MIN = 0.80

# First require Q > Q_MIN
eligible_states = np.flatnonzero(core_q > Q_MIN)

if len(eligible_states) == 0:
    # Guarantee that one native-like state is always assigned
    native_candidates = np.array(
        [np.argmax(core_q)],
        dtype=int,
    )

    print(
        f"WARNING: No macrostate has Q > {Q_MIN:.2f}. "
        "Using the highest-Q macrostate as the only native candidate."
    )
else:
    # Sort eligible states from highest to lowest Q
    eligible_by_q = eligible_states[
        np.argsort(core_q[eligible_states])[::-1]
    ]

    # Retain at most the two highest-Q eligible states
    native_candidates = eligible_by_q[:2]

print("\nNative candidates:")
for macro_idx in native_candidates:
    print(
        f"Macrostate {macro_idx}: "
        f"Q={core_q[macro_idx]:.3f}, "
        f"G={core_g[macro_idx]:.4f}"
    )

# Among the candidates, select the state with the smallest G.
# Higher Q resolves an exact tie in G.
native_old_index = min(
    native_candidates,
    key=lambda idx: (
        core_g[idx],
        -core_q[idx],
    ),
)
native_old_index = int(native_old_index)

# Sort all non-native macrostates by their core Q
non_native_old_indices = sorted(
    (
        macro_idx
        for macro_idx in range(n_macrostates)
        if macro_idx != native_old_index
    ),
    key=lambda idx: core_q[idx],
)

# Place the native state last so it receives the largest new index
macro_state_sorted = np.asarray(
    non_native_old_indices + [native_old_index],
    dtype=int,
)

# Construct the old-to-new macrostate index mapping
macro_state_mapping = {
    int(old_index): new_index
    for new_index, old_index in enumerate(macro_state_sorted)
}

native_index = macro_state_mapping[native_old_index]

print("\nMacrostate remapping:")
for old_index in macro_state_sorted:
    old_index = int(old_index)

    print(
        f"Macrostate old: {old_index} "
        f"-> New: {macro_state_mapping[old_index]}"
    )

print(
    f"\nSelected native-like macrostate: "
    f"old {native_old_index} -> new {native_index}; "
    f"Q={core_q[native_old_index]:.3f}, "
    f"G={core_g[native_old_index]:.4f}"
)

# convert macrostate_assignment to the sorted macrostate_assignment
macrostate_assignment_sorted = np.zeros_like(macrostate_assignment)
for i in range(len(macrostate_assignment)):
    macrostate_assignment_sorted[i] = macro_state_mapping[macrostate_assignment[i]]

# convert the microstates to macrostates
meta_dtrajs = [[macrostate_assignment_sorted[i] for i in dtraj] for dtraj in discrete_trajectories]
meta_dtrajs_flat = np.array(np.array(meta_dtrajs).flatten(), dtype=int)

# Check state probability along the whole trajectory
for i in range(n_macrostates):
    print(f"Macrostate {i}: {len(meta_dtrajs_flat[meta_dtrajs_flat==i])/len(meta_dtrajs_flat):.3f}")

In [ ]:
# fig = plot_qg_msm_nature(raw_data_all, meta_dtrajs_flat, nbins=75)
# outbase="QG_MSM2"
# fig.savefig(f"{outbase}.pdf")
# fig.savefig(f"{outbase}.png", dpi=600)

# Adjust bins=(Q bins, G bins). Fewer G bins make very sparse G sampling clearer.
fig = plot_qg_binned(
    raw_data_all,
    meta_dtrajs_flat,
    bins=(100, 40),
    q_range=(0.0, 1.0),
    min_count=1,
    max_free_energy=12.0
)
outbase="QG_MSM2"
fig.savefig(f"{outbase}.pdf", bbox_inches="tight")
fig.savefig(f"{outbase}.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# Check state probability along the last 200ns of the trajectory
meta_dtrajs_all = np.asarray(meta_dtrajs)
meta_dtrajs_last_200ns = meta_dtrajs_all[:,-2666:]
meta_dtrajs_last_200ns_flat = np.concatenate(meta_dtrajs_last_200ns)
for i in range(n_macrostates):
    print(f"Macrostate {i}: {len(meta_dtrajs_last_200ns_flat[meta_dtrajs_last_200ns_flat==i])/len(meta_dtrajs_last_200ns_flat):.3f}")

print("---------------\nNative state: \n")
native_index = np.max(meta_dtrajs_all)
print(f"{native_index=}")
native_prob = len(meta_dtrajs_last_200ns_flat[meta_dtrajs_last_200ns_flat==native_index])/len(meta_dtrajs_last_200ns_flat)
print(f"Native: {native_prob:.3f}, misfolded: {1-native_prob:.3f}")

# Save the results

In [ ]:
# Check if 'msm_results' folder exists, create if not
results_dir = 'msm_results'
os.makedirs(results_dir, exist_ok=True)

# Save arrays
np.savez(os.path.join(results_dir, 'msm_analysis_results.npz'),
         cluster_centers_scaled=cluster_centers_scaled,
         cluster_centers_original=cluster_centers_original,
         discrete_trajectories=discrete_trajectories,
         macrostate_assignment=macrostate_assignment,
         macrostate_assignment_sorted=macrostate_assignment_sorted,
         membership=membership,
         processed_data=processed_data,
         meta_dtrajs=meta_dtrajs
)

if scaler is not None:
    with open(os.path.join(results_dir, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)

if kmeans_model is not None:
    with open(os.path.join(results_dir, 'kmeans_model.pkl'), 'wb') as f:
        pickle.dump(kmeans_model, f)

if msm is not None:
    with open(os.path.join(results_dir, 'msm_model.pkl'), 'wb') as f:
        pickle.dump(msm, f)

if pcca is not None:
    with open(os.path.join(results_dir, 'pcca.pkl'), 'wb') as f:
        pickle.dump(pcca, f)
# if the BayesianMSM is used, save the models
if 'models' in locals() and models is not None:
    with open(os.path.join(results_dir, 'msm_models.pkl'), 'wb') as f:
        pickle.dump(models, f)